# Create the pipeline files

## Go to the project and create the folders

In [1]:
import os
from pathlib import Path
PROJECT = Path.home() / "Desktop" / "Project" / "sales-forecasting-assistant"
os.chdir(PROJECT)
for d in ["src/transform", "src/models", "dags"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print(os.getcwd())

/Users/dayoawe1/Desktop/Project/sales-forecasting-assistant


## Update the loader so it works from any folder

In [2]:
%%writefile src/ingest/load_raw.py
"""Load the Kaggle Store Sales CSVs into Snowflake SALES_DB.RAW."""
import os
from pathlib import Path
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = Path(__file__).resolve().parents[2] / "data" / "raw"
FILES = {
    "train.csv": "TRAIN",
    "test.csv": "TEST",
    "stores.csv": "STORES",
    "oil.csv": "OIL",
    "holidays_events.csv": "HOLIDAYS_EVENTS",
    "transactions.csv": "TRANSACTIONS",
}


def get_connection():
    return snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        private_key_file=os.path.expanduser(os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]),
        role="SYSADMIN",
        warehouse="SALES_WH",
        database="SALES_DB",
        schema="RAW",
    )


def main():
    conn = get_connection()
    try:
        for filename, table in FILES.items():
            df = pd.read_csv(DATA_DIR / filename, dtype=str)
            df.columns = [c.upper() for c in df.columns]
            success, _, nrows, _ = write_pandas(
                conn, df, table_name=table,
                auto_create_table=True, overwrite=True, quote_identifiers=False,
            )
            print(f"{table:<16} {'OK' if success else 'FAILED'}  {nrows:>10,} rows")
            if not success:
                raise RuntimeError(f"Loading {table} failed")
    finally:
        conn.close()


if __name__ == "__main__":
    main()

Overwriting src/ingest/load_raw.py


## Run the staging SQL from Python

In [3]:
%%writefile src/transform/run_staging.py
"""Run sql/03_staging.sql in Snowflake to rebuild the STAGING layer."""
from pathlib import Path
from src.ingest.load_raw import get_connection

SQL_FILE = Path(__file__).resolve().parents[2] / "sql" / "03_staging.sql"


def main():
    conn = get_connection()
    try:
        cursors = conn.execute_string(SQL_FILE.read_text())
        print(f"Staging built: {len(cursors)} statements executed")
    finally:
        conn.close()


if __name__ == "__main__":
    main()

Writing src/transform/run_staging.py


## Feature building, now including next week

In [4]:
%%writefile src/features/build_features.py
"""Build forecasting features with PySpark, including a row for next week."""
from datetime import timedelta
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window

INPUT_COLS = ["STORE_NBR", "FAMILY", "WEEK_START", "STORE_TYPE", "STORE_CLUSTER", "SALES",
              "PROMO_ITEMS", "HOLIDAY_DAYS", "AVG_OIL_PRICE", "STORE_TRANSACTIONS", "IS_FUTURE"]


def get_spark():
    return (SparkSession.builder
            .appName("sales-features")
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")
            .getOrCreate())


def build_features(sdf):
    """Lag, rolling and calendar features per store x family.
    Rolling windows use only PAST weeks (rows -N to -1) to prevent leakage."""
    w = Window.partitionBy("STORE_NBR", "FAMILY").orderBy("WEEK_START")
    for k in [1, 2, 4, 52]:
        sdf = sdf.withColumn(f"SALES_LAG_{k}", F.lag("SALES", k).over(w))
    return (sdf
            .withColumn("SALES_ROLL_MEAN_4",  F.avg("SALES").over(w.rowsBetween(-4, -1)))
            .withColumn("SALES_ROLL_MEAN_12", F.avg("SALES").over(w.rowsBetween(-12, -1)))
            .withColumn("SALES_ROLL_STD_12",  F.stddev("SALES").over(w.rowsBetween(-12, -1)))
            .withColumn("WEEK_OF_YEAR", F.weekofyear("WEEK_START"))
            .withColumn("MONTH", F.month("WEEK_START"))
            .withColumn("YEAR", F.year("WEEK_START")))


def load_weekly_with_future():
    from src.ingest.load_raw import get_connection
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM SALES_DB.STAGING.SALES_WEEKLY")
        pdf = cur.fetch_pandas_all()
        cur.execute("SELECT STORE_NBR, DATE FROM SALES_DB.STAGING.STORE_HOLIDAYS")
        hol = cur.fetch_pandas_all()
    finally:
        conn.close()

    pdf["WEEK_START"] = pd.to_datetime(pdf["WEEK_START"])
    # drop partial first/last weeks
    pdf = pdf[(pdf["WEEK_START"] > pdf["WEEK_START"].min()) &
              (pdf["WEEK_START"] < pdf["WEEK_START"].max())].copy()
    pdf["IS_FUTURE"] = 0

    # one future row per series for the week we want to forecast
    next_week = pdf["WEEK_START"].max() + timedelta(days=7)
    future = pdf.sort_values("WEEK_START").groupby(["STORE_NBR", "FAMILY"]).tail(1).copy()
    future["WEEK_START"] = next_week
    future["SALES"] = float("nan")
    future["STORE_TRANSACTIONS"] = float("nan")
    future["IS_FUTURE"] = 1
    # PROMO_ITEMS: assume next week's promotions match last week's (planned promos would replace this)
    hol["DATE"] = pd.to_datetime(hol["DATE"])
    in_week = hol[(hol["DATE"] >= next_week) & (hol["DATE"] < next_week + timedelta(days=7))]
    future["HOLIDAY_DAYS"] = future["STORE_NBR"].map(in_week.groupby("STORE_NBR").size()).fillna(0)

    return pd.concat([pdf, future], ignore_index=True)[INPUT_COLS]


def main():
    from snowflake.connector.pandas_tools import write_pandas
    from src.ingest.load_raw import get_connection

    pdf = load_weekly_with_future()
    spark = get_spark()
    try:
        features = build_features(spark.createDataFrame(pdf)).toPandas()
    finally:
        spark.stop()
    features["WEEK_START"] = pd.to_datetime(features["WEEK_START"]).dt.date

    conn = get_connection()
    try:
        ok, _, n, _ = write_pandas(conn, features, table_name="FEATURES",
                                   database="SALES_DB", schema="ANALYTICS",
                                   auto_create_table=True, overwrite=True,
                                   quote_identifiers=False)
    finally:
        conn.close()
    print(f"FEATURES {'OK' if ok else 'FAILED'} {n:,} rows")
    if not ok:
        raise RuntimeError("Writing FEATURES failed")


if __name__ == "__main__":
    main()

Overwriting src/features/build_features.py


## Model evaluation (quality gate) and forecasting

In [5]:
%%writefile src/models/train_forecast.py
"""Evaluate the XGBoost model (quality gate) and forecast next week."""
import os
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import xgboost as xgb
from snowflake.connector.pandas_tools import write_pandas
from src.ingest.load_raw import get_connection

FEATURES = ["SALES_LAG_1", "SALES_LAG_2", "SALES_LAG_4", "SALES_LAG_52",
            "SALES_ROLL_MEAN_4", "SALES_ROLL_MEAN_12", "SALES_ROLL_STD_12",
            "PROMO_ITEMS", "HOLIDAY_DAYS", "WEEK_OF_YEAR", "MONTH",
            "STORE_NBR", "STORE_CLUSTER", "FAMILY_CODE", "STORE_TYPE_CODE"]
MODEL_NAME = "xgboost_global"
WAPE_THRESHOLD = float(os.environ.get("WAPE_THRESHOLD", "0.12"))


def run_ts():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def load_features():
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM SALES_DB.ANALYTICS.FEATURES")
        df = cur.fetch_pandas_all()
    finally:
        conn.close()
    df["WEEK_START"] = pd.to_datetime(df["WEEK_START"])
    num = [c for c in df.columns if c not in ("FAMILY", "STORE_TYPE", "WEEK_START")]
    df[num] = df[num].apply(pd.to_numeric, errors="coerce")
    df["FAMILY_CODE"] = df["FAMILY"].astype("category").cat.codes
    df["STORE_TYPE_CODE"] = df["STORE_TYPE"].astype("category").cat.codes
    return df


def train_model(train):
    model = xgb.XGBRegressor(n_estimators=400, max_depth=8, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             tree_method="hist", n_jobs=-1)
    model.fit(train[FEATURES], np.log1p(train["SALES"].clip(lower=0)))
    return model


def predict(model, rows):
    return np.expm1(model.predict(rows[FEATURES])).clip(0)


def write(df, table):
    conn = get_connection()
    try:
        ok, _, n, _ = write_pandas(conn, df, table_name=table, database="SALES_DB",
                                   schema="ANALYTICS", auto_create_table=True,
                                   overwrite=False, quote_identifiers=False)
    finally:
        conn.close()
    if not ok:
        raise RuntimeError(f"Writing {table} failed")
    print(f"{table}: appended {n:,} rows")


def evaluate():
    """Hold out the latest actual week, train on earlier weeks, and check WAPE."""
    df = load_features()
    actual = df[df["IS_FUTURE"] == 0]
    eval_week = actual["WEEK_START"].max()
    train = actual[(actual["WEEK_START"] < eval_week) & actual["SALES_LAG_52"].notna()]
    test = actual[actual["WEEK_START"] == eval_week]

    pred = predict(train_model(train), test)
    wape = float(np.abs(test["SALES"].values - pred).sum() / np.abs(test["SALES"].values).sum())

    write(pd.DataFrame([{"RUN_TS": run_ts(), "MODEL": MODEL_NAME,
                         "EVAL_WEEK": str(eval_week.date()), "WAPE": wape,
                         "N_SERIES": int(len(test)), "THRESHOLD": WAPE_THRESHOLD,
                         "PASSED": wape <= WAPE_THRESHOLD}]), "ACCURACY_LOG")
    print(f"Holdout week {eval_week.date()}: WAPE = {wape:.4f} (threshold {WAPE_THRESHOLD})")
    if wape > WAPE_THRESHOLD:
        raise ValueError(f"Quality gate failed: WAPE {wape:.4f} > {WAPE_THRESHOLD}")


def forecast():
    """Train on all actual weeks and forecast the next week for every series."""
    df = load_features()
    train = df[(df["IS_FUTURE"] == 0) & df["SALES_LAG_52"].notna()]
    future = df[df["IS_FUTURE"] == 1]
    if future.empty:
        raise ValueError("No future rows found in FEATURES")

    out = pd.DataFrame({"STORE_NBR": future["STORE_NBR"].values,
                        "FAMILY": future["FAMILY"].values,
                        "WEEK_START": future["WEEK_START"].dt.date.values,
                        "FORECAST": predict(train_model(train), future),
                        "MODEL": MODEL_NAME,
                        "RUN_TS": run_ts()})
    write(out, "FORECASTS")
    print(f"Forecast week {out['WEEK_START'].iloc[0]}: total = {out['FORECAST'].sum():,.0f}")

Writing src/models/train_forecast.py


## The Airflow DAG (the pipeline definition)

In [6]:
%%writefile dags/sales_forecasting_dag.py
"""Weekly sales forecasting pipeline: Snowflake -> PySpark -> XGBoost."""
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator


def _load_raw():
    from src.ingest.load_raw import main
    main()


def _build_staging():
    from src.transform.run_staging import main
    main()


def _build_features():
    from src.features.build_features import main
    main()


def _evaluate_model():
    from src.models.train_forecast import evaluate
    evaluate()


def _forecast_next_week():
    from src.models.train_forecast import forecast
    forecast()


default_args = {"owner": "dayo", "retries": 1, "retry_delay": timedelta(minutes=5)}

with DAG(
    dag_id="sales_forecasting_pipeline",
    description="Load -> stage -> features -> quality gate -> forecast",
    start_date=datetime(2026, 10, 1),
    schedule="0 6 * * 1",          # every Monday at 06:00 UTC
    catchup=False,
    default_args=default_args,
    tags=["forecasting", "snowflake", "spark", "xgboost"],
) as dag:
    load_raw = PythonOperator(task_id="load_raw", python_callable=_load_raw)
    build_staging = PythonOperator(task_id="build_staging", python_callable=_build_staging)
    build_features = PythonOperator(task_id="build_features", python_callable=_build_features)
    evaluate_model = PythonOperator(task_id="evaluate_model", python_callable=_evaluate_model)
    forecast_next_week = PythonOperator(task_id="forecast_next_week",
                                        python_callable=_forecast_next_week)

    load_raw >> build_staging >> build_features >> evaluate_model >> forecast_next_week

Writing dags/sales_forecasting_dag.py


## Docker files

In [7]:
Path("Dockerfile").write_text("""FROM apache/airflow:2.10.5-python3.11
USER root
RUN apt-get update && apt-get install -y --no-install-recommends default-jre-headless \\
    && rm -rf /var/lib/apt/lists/*
ENV JAVA_HOME=/usr/lib/jvm/default-java
USER airflow
COPY requirements-airflow.txt /requirements-airflow.txt
RUN pip install --no-cache-dir -r /requirements-airflow.txt
""")

Path("requirements-airflow.txt").write_text("pyspark==3.5.3\nxgboost==2.1.1\npython-dotenv\n")

Path("docker-compose.yml").write_text("""services:
  airflow:
    build: .
    image: sales-airflow:latest
    command: standalone
    ports:
      - "8080:8080"
    environment:
      AIRFLOW__CORE__LOAD_EXAMPLES: "false"
      PYTHONPATH: /opt/airflow/project
      SNOWFLAKE_ACCOUNT: ${SNOWFLAKE_ACCOUNT}
      SNOWFLAKE_USER: ${SNOWFLAKE_USER}
      SNOWFLAKE_PRIVATE_KEY_FILE: /opt/airflow/keys/rsa_key.p8
      WAPE_THRESHOLD: "0.12"
    volumes:
      - ./dags:/opt/airflow/dags
      - ./:/opt/airflow/project
      - ${HOME}/.snowflake/rsa_key.p8:/opt/airflow/keys/rsa_key.p8:ro
""")

Path(".dockerignore").write_text("data/\n.git/\nnotebooks/\n.env\n**/.ipynb_checkpoints\n**/__pycache__\n")
print("Docker files written")

Docker files written


## Test the modules locally before Docker

In [8]:
import subprocess, sys
os.environ["JAVA_HOME"] = subprocess.check_output(["/usr/libexec/java_home", "-v", "17"]).decode().strip()
sys.path.insert(0, str(PROJECT))

In [9]:
from src.transform.run_staging import main as staging; staging()

Staging built: 14 statements executed


In [10]:
from src.features.build_features import main as feats; feats()

26/10/06 11:37:54 WARN Utils: Your hostname, Dayos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.154 instead (on interface en0)
26/10/06 11:37:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 11:37:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/06 11:37:58 WARN TaskSetManager: Stage 0 contains a task of very large size (7102 KiB). The maximum recommended task size is 1000 KiB.


FEATURES OK 429,462 rows


In [11]:
from src.models.train_forecast import evaluate, forecast
evaluate()
forecast()

ACCURACY_LOG: appended 1 rows
Holdout week 2017-08-07: WAPE = 0.0999 (threshold 0.12)
FORECASTS: appended 1,782 rows
Forecast week 2017-08-14: total = 5,588,580
